In [ ]:
# Import the CSV loader
from langchain_community.document_loaders.csv_loader import CSVLoader

# Create a document loader
loader = CSVLoader("fifa_countries_audience.csv")

# Load the CSV file
data = loader.load()

# Print the first document
print(data[0])

In [ ]:
# Import library
from langchain_community.document_loaders import PyPDFLoader

# Create a document loader
loader = PyPDFLoader("rag_vs_fine_tuning.pdf")

# Load the document
data = loader.load()

print(data[0])

In [ ]:
from langchain_community.document_loaders import UnstructuredHTMLLoader

# Create a document loader for an HTML file
loader = UnstructuredHTMLLoader("index.html")

# Load the document
data = loader.load()

# Print the first document
print(data[0])

# Print the first document's metadata
print(data[0].metadata)

In [1]:
# spliting doucments
# Import the character splitter
from langchain_text_splitters import CharacterTextSplitter

quote = 'Words are flowing out like endless rain into a paper cup,\nthey slither while they pass,\nthey slip away across the universe.'
chunk_size = 24
chunk_overlap = 10

# Create an instance of the splitter class
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=24,
    chunk_overlap=10)

# Split the string and print the chunks
docs = splitter.split_text(quote)
print(docs)
print([len(doc) for doc in docs])

ModuleNotFoundError: No module named 'langchain_text_splitters'

In [2]:
#better ways
from langchain_text_splitters import RecursiveCharacterTextSplitter

quote = 'Words are flowing out like endless rain into a paper cup,\nthey slither while they pass,\nthey slip away across the universe.'
chunk_size = 24
chunk_overlap = 10

# Create an instance of the splitter class
splitter = RecursiveCharacterTextSplitter(
    separators=["\n", " ", ""],
    chunk_size=24,
    chunk_overlap=10)

# Split the document and print the chunks
docs = splitter.split_text(quote)
print(docs)
print([len(doc) for doc in docs])

ModuleNotFoundError: No module named 'langchain_text_splitters'

In [ ]:

import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# ==========================================
# 1. Load the PDF document
# ==========================================

loader = PyPDFLoader("rag_vs_fine_tuning.pdf")
data = loader.load()

# ==========================================
# 2. Split the document into chunks
# ==========================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.split_documents(data)

# ==========================================
# 3. Create the OpenAI embedding model
# ==========================================

embedding_function = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# ==========================================
# 4. Store document chunks in Chroma
# ==========================================

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_function,
    persist_directory=os.path.join(os.getcwd(), "chroma_db")
)

# ==========================================
# 5. Configure Chroma as a retriever
# ==========================================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# ==========================================
# 6. Define the RAG prompt
# ==========================================

message = """
Answer the following question using the context provided.

Context:
{context}

Question:
{question}

Answer:
"""

# ==========================================
# 7. Create the chat prompt template
# ==========================================

prompt_template = ChatPromptTemplate.from_messages([
    ("human", message)
])

# ==========================================
# 8. Initialize the LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# ==========================================
# 9. Format retrieved documents
# ==========================================

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ==========================================
# 10. Build the RAG chain
# ==========================================

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
)

# ==========================================
# 11. Ask a question
# ==========================================

question = "Which popular LLMs were considered in the paper?"

response = rag_chain.invoke(question)

print(response.content)
